In [ ]:
import os
import json
import pandas as pd
from PIL import Image, ImageDraw, ImageFont

# --- GOOGLE DRIVE API SETUP ---
from google.oauth2.service_account import Credentials
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload

# Set your Google Drive Folder ID where images should be saved
GDRIVE_FOLDER_ID = "1hNIJLf32K8pHoRBbg7lZxHWhYcb4yKQC"

creds_raw = os.getenv("GDRIVE_CREDENTIALS")

if creds_raw:
    creds_json = json.loads(creds_raw)
    scopes = ["https://www.googleapis.com/auth/drive"]
    creds = Credentials.from_service_account_info(creds_json, scopes=scopes)
    drive_service = build("drive", "v3", credentials=creds)
    print("Successfully connected to Google Drive API.")
else:
    print("Warning: GDRIVE_CREDENTIALS environment variable not set.")
    drive_service = None

# --- HELPER FUNCTIONS FOR GDRIVE ---
def upload_file(local_path, folder_id, mime_type="image/png"):
    """Uploads a file to Google Drive and transfers ownership to personal account."""
    if not drive_service:
        print("Skipping upload: Drive service not initialized.")
        return
    file_metadata = {
        "name": os.path.basename(local_path),
        "parents": [folder_id]
    }
    media = MediaFileUpload(local_path, mimetype=mime_type)
    file = drive_service.files().create(
        body=file_metadata,
        media_body=media,
        fields="id",
        supportsAllDrives=True
    ).execute()
    file_id = file.get("id")
    print(f"Uploaded {os.path.basename(local_path)} (ID: {file_id})")

    # Transfer ownership to personal account to bypass service account quota limits
    try:
        permission = {
            'type': 'user',
            'role': 'owner',
            'emailAddress': 'buisness.foxnox@gmail.com'
        }
        drive_service.permissions().create(
            fileId=file_id,
            body=permission,
            transferOwnership=True,
            supportsAllDrives=True
        ).execute()
        print("Transferred ownership to buisness.foxnox@gmail.com")
    except Exception as e:
        print(f"Note on ownership transfer: {e}")

# --- SETUP LOCAL WORK DIRECTORIES ---
LOCAL_DATA_DIR = "./data"
LOCAL_TEMPLATE_DIR = "./data/templates"
os.makedirs(LOCAL_TEMPLATE_DIR, exist_ok=True)

today = pd.Timestamp.today().strftime('%d/%m/%y')

# --- FONT HELPER ---
def get_font(size):
    try:
        return ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", size)
    except:
        return ImageFont.load_default()

# --- TEMPLATE GENERATION FUNCTION ---
def create_insta_template(text, output_filename, logo_path=None):
    width, height = 1080, 1440
    img = Image.new('RGBA', (width, height), (255, 255, 255, 255))
    draw = ImageDraw.Draw(img)
    navy_blue, black = (26, 54, 93), (0, 0, 0)
    inner_margin = 45

    draw.rectangle([20, 20, width - 20, height - 20], outline=navy_blue, width=1)
    draw.rectangle([inner_margin, inner_margin, width - inner_margin, height - inner_margin], outline=navy_blue, width=4)

    def draw_sparkle(draw_obj, center_x, center_y, size_px, color):
        points = [
            (center_x, center_y - size_px),
            (center_x + size_px/4, center_y - size_px/4),
            (center_x + size_px, center_y),
            (center_x + size_px/4, center_y + size_px/4),
            (center_x, center_y + size_px),
            (center_x - size_px/4, center_y + size_px/4),
            (center_x - size_px, center_y),
            (center_x - size_px/4, center_y - size_px/4)
        ]
        draw_obj.polygon(points, fill=color)

    draw_sparkle(draw, 930, 100, 25, navy_blue)
    draw_sparkle(draw, 960, 80, 12, navy_blue)
    draw_sparkle(draw, 120, height - 100, 25, navy_blue)
    draw_sparkle(draw, 90, height - 80, 12, navy_blue)

    left, right = inner_margin + 20, width - inner_margin - 20
    usable_width = right - left
    font_size = 85

    while font_size >= 30:
        font_header = get_font(font_size)
        bbox = draw.textbbox((0, 0), text, font=font_header)
        text_width = bbox[2] - bbox[0]
        if text_width <= usable_width:
            break
        font_size -= 2

    x = left + (usable_width - text_width) / 2
    y = 220
    draw.text((x, y), text, fill=black, font=font_header)

    if logo_path and os.path.exists(logo_path):
        logo = Image.open(logo_path).convert("RGBA")
        logo_w = 180
        aspect_ratio = logo.size[1] / logo.size[0]
        logo_h = int(logo_w * aspect_ratio)
        logo = logo.resize((logo_w, logo_h), Image.Resampling.LANCZOS)
        padding = 5
        logo_x = width - inner_margin - logo_w - padding
        logo_y = height - inner_margin - logo_h - padding
        img.paste(logo, (logo_x, logo_y), logo)

    output_path = os.path.join(LOCAL_TEMPLATE_DIR, output_filename)
    img.save(output_path)
    print(f"Generated locally: {output_path}")
    
    # Upload generated image to Google Drive
    if GDRIVE_FOLDER_ID != "YOUR_GOOGLE_DRIVE_FOLDER_ID":
        upload_file(output_path, GDRIVE_FOLDER_ID)

# --- EXECUTE TEMPLATE GENERATION ---
# Look directly in root directory for Logo_PNG.png
my_logo_path = 'Logo_PNG.png'

templates_to_create = [
    (f"BAN {today}", "BAN TEMPLATE.png"),
    (f"RESULTS {today}", "RESULTS TEMPLATE.png"),
    ("GLOBAL MARKETS", "GLOBAL TEMPLATE.png"),
    (f"TOP 10 MARKET MOVERS ({today})", "TOP 10 TEMPLATE.png"),
    (f"SECTORAL INDICES ({today})", "SECTORAL INDICES.png"),
    (f"MAJOR INDICES ({today})", "MAJOR INDICES.png"),
    ("SUPPORT RESISTANCE (INDIA VIX BASED)", "INDIA VIX.png"),
    (f"52 WEEK HIGH LOW ({today})", "52WHL.png"),
    (f"ADVANCE DECLINE RATIO ({today})", "ADR.png"),
    ("NIFTY OPEN INTEREST TOP 5 STRIKES\nCALL AND PUT SIDE WITHIN 3% RANGE", "OI.png"),
    ("NIFTY OTM CALL PUT OPEN INTEREST\nT AND T-1 TRADING DAYS BIFURCATION", "PCR.png"),
    (f"DELIVERY HEAVY STOCKS ({today})", "DELIVERY.png")
]

for title_text, file_name in templates_to_create:
    create_insta_template(title_text, file_name, logo_path=my_logo_path)